In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


In [27]:
train = pd.read_pickle('../datos/entrenamiento/train_transformado.pkl')

# Preselcción de variables

In [28]:
def correlaciones_fuertes(df, lim_inf = 0.3, lim_sup = 1,drop_dupli=True):
    #Calcula la matriz de correlación
    c = df.corr().abs()
    #Lo pasa todo a filas
    c= c.unstack()
    #Pasa el índice a columnas y le pone nombres
    c = pd.DataFrame(c).reset_index()
    c.columns = ['var1','var2','corr']
    #A dataframe, filtra limites y ordena en descendiente
    c = c.loc[(c['corr'] > lim_inf) &  (c['corr'] < lim_sup),:].sort_values(by = 'corr', ascending=False)
    #Desduplica las correlaciones (o no si drop_dupli es False)
    c = c if drop_dupli == False else c.drop_duplicates(subset = ['corr'])
    #Devuelve la salida
    return(c)

In [29]:
correlaciones_fuertes(train)

,var1,var2,corr
98,tasa_interes,calificacion_prestamo,0.933330
9,edad,duracion_credito,0.876804
202,propiedad_vivienda_MORTGAGE,propiedad_vivienda_RENT,0.855884
137,porcentaje_ingreso,monto_prestamo,0.647602
65,calificacion_prestamo,incumplimiento_historial,0.536140
157,incumplimiento_historial,tasa_interes,0.500142
23,ingreso,monto_prestamo,0.383795
63,calificacion_prestamo,estado_prestamo,0.379977
121,estado_prestamo,porcentaje_ingreso,0.379368
101,tasa_interes,estado_prestamo,0.340751


In [30]:
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier

x = train.drop(columns=['estado_prestamo'])
y = train['estado_prestamo']

xgb = XGBClassifier(n_jobs = -1, eval_metric='auc')

xgb.fit(x,y)

permutacion = permutation_importance(xgb, 
                                     x, y, 
                                     scoring = 'roc_auc',
                                     n_repeats=5, n_jobs = -1)

In [31]:
def ranking_per(predictoras,permutacion):
    ranking_per = pd.DataFrame({'variable': predictoras.columns, 'importancia_per': permutacion.importances_mean}).sort_values(by = 'importancia_per', ascending = False)
    ranking_per['ranking_per'] = np.arange(0,ranking_per.shape[0])
    return(ranking_per)

In [32]:
ranking_per(x, permutacion)

,variable,importancia_per,ranking_per
1,ingreso,0.108824,0
6,porcentaje_ingreso,0.098734,1
3,calificacion_prestamo,0.071706,2
11,propiedad_vivienda_RENT,0.045372,3
10,propiedad_vivienda_OWN,0.026819,4
5,tasa_interes,0.022784,5
4,monto_prestamo,0.020205,6
2,duracion_empleo,0.014978,7
17,intencion_prestamo_VENTURE,0.014381,8
0,edad,0.014376,9


Eliminar las correlaciones fuertes en base a permutation importance

In [33]:
train.drop(columns=['tasa_interes','duracion_credito','propiedad_vivienda_MORTGAGE'], inplace=True)

In [34]:
def ranking_mi(mutual_selector, modo = 'tabla'):
    #Maqueta el ranking
    ranking_mi = pd.DataFrame(mutual_selector, index = x.columns).reset_index()
    ranking_mi.columns = ['variable','importancia_mi']
    ranking_mi = ranking_mi.sort_values(by = 'importancia_mi', ascending = False)
    ranking_mi['ranking_mi'] = np.arange(0,ranking_mi.shape[0])
    #Muestra la salida
    if modo == 'tabla':
        return(ranking_mi)
    else:
        g = ranking_mi[0:15].importancia_mi.sort_values().plot.barh()
        g.set_yticklabels(ranking_mi[0:15].sort_values(by = 'importancia_mi').variable)
        return(g)

In [35]:
x = train.drop(columns=['estado_prestamo'])
y = train['estado_prestamo']



In [36]:
from sklearn.feature_selection import mutual_info_classif

mutual_selector = mutual_info_classif(x,y)
ranking_mi(mutual_selector)



,variable,importancia_mi,ranking_mi
1,ingreso,0.109487,0
5,porcentaje_ingreso,0.083960,1
3,calificacion_prestamo,0.077950,2
8,propiedad_vivienda_RENT,0.027166,3
4,monto_prestamo,0.019682,4
6,incumplimiento_historial,0.012251,5
7,propiedad_vivienda_OWN,0.009851,6
2,duracion_empleo,0.009703,7
10,intencion_prestamo_EDUCATION,0.005363,8
14,intencion_prestamo_VENTURE,0.004713,9


In [37]:
from sklearn.feature_selection import RFECV
from xgboost import XGBClassifier

rfe = RFECV(estimator = XGBClassifier(n_jobs = -1, eval_metric='auc'),
            cv = 3,
            scoring = 'roc_auc',
            n_jobs = -1)

rfe.fit(x,y)

rank_rfe = pd.DataFrame({'variable': x.columns, 'ranking_rfe': rfe.ranking_}).sort_values(by = 'ranking_rfe')
rank_rfe

,variable,ranking_rfe
1,ingreso,1
3,calificacion_prestamo,1
7,propiedad_vivienda_OWN,1
5,porcentaje_ingreso,1
12,intencion_prestamo_MEDICAL,1
11,intencion_prestamo_HOMEIMPROVEMENT,1
9,intencion_prestamo_DEBTCONSOLIDATION,1
8,propiedad_vivienda_RENT,1
14,intencion_prestamo_VENTURE,1
2,duracion_empleo,2


In [38]:
train.drop(columns=['incumplimiento_historial','intencion_prestamo_PERSONAL'], inplace=True)

# Balanceo de clases

In [39]:
train.estado_prestamo.value_counts()

estado_prestamo
0    21060
1     6029
Name: count, dtype: int64

No aplicamos balanceo de clases

In [40]:
train.to_pickle('../datos/entrenamiento/train_preseleccion.pkl')